# Phase B: Modèles de Prédiction Complets


In [20]:
import pandas as pd
import numpy as np
import pickle
import os
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

In [21]:
# Load prepared data
df = pd.read_parquet("../Data/prepared_reviews.parquet")
print(f"Dataset shape: {df.shape}")

Dataset shape: (1000000, 4)


In [39]:
# Sample for training
SAMPLE_SIZE = 100
df_sample = df.sample(min(SAMPLE_SIZE, len(df)), random_state=42)
print(f"Using {len(df_sample)} samples")

# Polarity mapping
polarity_map = {'negative': 0, 'neutral': 1, 'positive': 2}
df_sample['polarity_num'] = df_sample['polarity'].map(polarity_map)

# Split
X = df_sample['text']
y = df_sample['polarity_num']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {len(X_train)}, Test: {len(X_test)}")

Using 100 samples
Train: 80, Test: 20


## 1. Représentation: N-Grammes (CountVectorizer)

In [40]:
# N-Gram Vectorizer (unigrams + bigrams)
ngram_vectorizer = CountVectorizer(max_features=5000, ngram_range=(1, 2), stop_words='english')
X_train_ngram = ngram_vectorizer.fit_transform(X_train)
X_test_ngram = ngram_vectorizer.transform(X_test)
print(f"N-Gram shape: {X_train_ngram.shape}")

N-Gram shape: (80, 5000)


## 2. Représentation: TF-IDF

In [41]:
# TF-IDF Vectorizer
tfidf_vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), stop_words='english')
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)
print(f"TF-IDF shape: {X_train_tfidf.shape}")

TF-IDF shape: (80, 5000)


## 3. Représentation: LLM Embeddings (Sentence Transformers)

In [42]:
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

# Remplacer par le modèle choisi
llm_model = SentenceTransformer('all-mpnet-base-v2')  

print("Generating LLM embeddings...")

X_train_llm = llm_model.encode(X_train.tolist(), show_progress_bar=True, batch_size=32)
X_test_llm = llm_model.encode(X_test.tolist(), show_progress_bar=True, batch_size=32)
print(f"LLM Embeddings shape: {X_train_llm.shape}")  

c:\Users\anas\Desktop\Cours\SAE\.venv\Lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\anas\.cache\huggingface\hub\models--sentence-transformers--all-mpnet-base-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 810.96it/s, Materializing param=pool

Generating LLM embeddings...


Batches: 100%|██████████| 1/1 [00:06<00:00,  6.44s/it]

LLM Embeddings shape: (80, 768)


---
# ML CLASSIQUE


In [43]:
# Store all results
results = []

def train_and_eval(model, name, X_tr, X_te, y_tr, y_te, representation):
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    acc = accuracy_score(y_te, y_pred)
    results.append({'Model': name, 'Representation': representation, 'Accuracy': acc})
    print(f"{name} ({representation}): {acc:.4f}")
    return model, y_pred

### 4.1 ML + N-Gram

In [44]:
print("=== ML + N-Gram ===")
lr_ngram, _ = train_and_eval(LogisticRegression(max_iter=500, n_jobs=-1), 'Logistic Regression', X_train_ngram, X_test_ngram, y_train, y_test, 'N-Gram')
svm_ngram, _ = train_and_eval(LinearSVC(max_iter=1000), 'SVM', X_train_ngram, X_test_ngram, y_train, y_test, 'N-Gram')
nb_ngram, _ = train_and_eval(MultinomialNB(), 'Naive Bayes', X_train_ngram, X_test_ngram, y_train, y_test, 'N-Gram')
rf_ngram, _ = train_and_eval(RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42), 'Random Forest', X_train_ngram, X_test_ngram, y_train, y_test, 'N-Gram')

=== ML + N-Gram ===


c:\Users\anas\Desktop\Cours\SAE\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


Logistic Regression (N-Gram): 0.6000
SVM (N-Gram): 0.6000
Naive Bayes (N-Gram): 0.6000
Random Forest (N-Gram): 0.6000


### 4.2 ML + TF-IDF

In [45]:
print("=== ML + TF-IDF ===")
lr_tfidf, _ = train_and_eval(LogisticRegression(max_iter=500), 'Logistic Regression', X_train_tfidf, X_test_tfidf, y_train, y_test, 'TF-IDF')
svm_tfidf, y_pred_best = train_and_eval(LinearSVC(max_iter=1000), 'SVM', X_train_tfidf, X_test_tfidf, y_train, y_test, 'TF-IDF')
nb_tfidf, _ = train_and_eval(MultinomialNB(), 'Naive Bayes', X_train_tfidf, X_test_tfidf, y_train, y_test, 'TF-IDF')
rf_tfidf, _ = train_and_eval(RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42), 'Random Forest', X_train_tfidf, X_test_tfidf, y_train, y_test, 'TF-IDF')

=== ML + TF-IDF ===
Logistic Regression (TF-IDF): 0.6000
SVM (TF-IDF): 0.6000
Naive Bayes (TF-IDF): 0.6000
Random Forest (TF-IDF): 0.6000


### 4.3 ML + LLM Embeddings

In [46]:
print("=== ML + LLM ===")
lr_llm, _ = train_and_eval(LogisticRegression(max_iter=500), 'Logistic Regression', X_train_llm, X_test_llm, y_train, y_test, 'LLM')
svm_llm, _ = train_and_eval(LinearSVC(max_iter=1000), 'SVM', X_train_llm, X_test_llm, y_train, y_test, 'LLM')
rf_llm, _ = train_and_eval(RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42), 'Random Forest', X_train_llm, X_test_llm, y_train, y_test, 'LLM')

=== ML + LLM ===
Logistic Regression (LLM): 0.7000
SVM (LLM): 0.8500
Random Forest (LLM): 0.6500


In [47]:
import joblib
joblib.dump(results, 'results_ml.pkl')

# Créer le dossier Sauvegarde s'il n'existe pas
save_dir = "../Sauvegarde"
os.makedirs(save_dir, exist_ok=True)

# --- Sauvegarde des résultats des modèles classiques (ML) ---
joblib.dump(results, os.path.join(save_dir, 'results_ml.pkl'))

# --- Sauvegarde des modèles classiques ---
# N-Gram
joblib.dump(lr_ngram, os.path.join(save_dir, 'lr_ngram.pkl'))
joblib.dump(svm_ngram, os.path.join(save_dir, 'svm_ngram.pkl'))
joblib.dump(nb_ngram, os.path.join(save_dir, 'nb_ngram.pkl'))
joblib.dump(rf_ngram, os.path.join(save_dir, 'rf_ngram.pkl'))

# TF-IDF
joblib.dump(lr_tfidf, os.path.join(save_dir, 'lr_tfidf.pkl'))
joblib.dump(svm_tfidf, os.path.join(save_dir, 'svm_tfidf.pkl'))
joblib.dump(nb_tfidf, os.path.join(save_dir, 'nb_tfidf.pkl'))
joblib.dump(rf_tfidf, os.path.join(save_dir, 'rf_tfidf.pkl'))

# LLM
joblib.dump(lr_llm, os.path.join(save_dir, 'lr_llm.pkl'))
joblib.dump(svm_llm, os.path.join(save_dir, 'svm_llm.pkl'))
joblib.dump(rf_llm, os.path.join(save_dir, 'rf_llm.pkl'))

# --- Sauvegarde des données de test (pour matrices de confusion) ---
np.save(os.path.join(save_dir, 'X_test_ngram.npy'), X_test_ngram.toarray())   # conversion en dense
np.save(os.path.join(save_dir, 'X_test_tfidf.npy'), X_test_tfidf.toarray())
np.save(os.path.join(save_dir, 'X_test_llm.npy'), X_test_llm)
joblib.dump(y_test, os.path.join(save_dir, 'y_test.pkl'))

print("Toutes les données et modèles ont été sauvegardés dans le dossier ../Sauvegarde/.")

Toutes les données et modèles ont été sauvegardés dans le dossier ../Sauvegarde/.


---
# 5 ) DEEP LEARNING 
## Architectures MLP 

In [48]:
import functools
import keras_tuner as kt
from tensorflow import keras
from tensorflow.keras import layers
import tensorflow as tf
import numpy as np
from sklearn.metrics import accuracy_score
import tempfile
import os

In [49]:
def build_model(hp, representation_type='sparse', input_dim=None, num_classes=3):
    if input_dim is None:
        input_dim = 5000 if representation_type == 'sparse' else 384

    model = keras.Sequential()
    model.add(layers.Input(shape=(input_dim,)))
    model.add(layers.BatchNormalization())

    if representation_type == 'llm':
        num_layers_range = (1, 4)
        units_range = (64, 512, 64)
        dropout_range = (0.2, 0.7)
        use_batchnorm = True
    else:
        num_layers_range = (1, 3)
        units_range = (128, 1024, 128)
        dropout_range = (0.2, 0.7)
        use_batchnorm = False

    num_layers = hp.Int('num_layers', min_value=num_layers_range[0], max_value=num_layers_range[1])

    for i in range(num_layers):
        units = hp.Int(f'units_{i}', 
                       min_value=units_range[0], 
                       max_value=units_range[1], 
                       step=units_range[2])
        activation = hp.Choice(f'activation_{i}', ['relu', 'tanh', 'elu'])
        l2_reg = hp.Float(f'l2_{i}', 1e-6, 1e-3, sampling='log', default=0)
        model.add(layers.Dense(units, activation=activation,
                               kernel_regularizer=keras.regularizers.l2(l2_reg)))

        if use_batchnorm:
            model.add(layers.BatchNormalization())

        dropout = hp.Float(f'dropout_{i}', 
                           min_value=dropout_range[0], 
                           max_value=dropout_range[1], 
                           step=0.1)
        model.add(layers.Dropout(dropout))

    # Couche de sortie avec num_classes
    model.add(layers.Dense(num_classes, activation='softmax'))

    lr = hp.Float('learning_rate', 1e-5, 1e-2, sampling='log')

    if representation_type == 'llm':
        optimizer_choice = hp.Choice('optimizer', ['adam', 'sgd', 'adamax'])
        if optimizer_choice == 'adam':
            optimizer = keras.optimizers.Adam(learning_rate=lr)
        elif optimizer_choice == 'sgd':
            optimizer = keras.optimizers.SGD(learning_rate=lr)
        else:
            optimizer = keras.optimizers.Adamax(learning_rate=lr)
    else:
        optimizer = keras.optimizers.Adam(learning_rate=lr)

    model.compile(
        optimizer=optimizer,
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

In [50]:
def tune_representation(X_train, X_test, y_train, y_test, representation_name, input_dim, representation_type='sparse', num_classes=3):
    print(f"\n{'='*50}")
    print(f"TUNING POUR REPRÉSENTATION: {representation_name}")
    print(f"{'='*50}")

    build_fn = functools.partial(build_model, representation_type=representation_type, input_dim=input_dim, num_classes=num_classes)

    # Changer le nom du projet pour distinguer la prédiction de note
    project_suffix = "_rating" if num_classes == 5 else ""
    tuner = kt.Hyperband(
        build_fn,
        objective='val_accuracy',
        max_epochs=20,
        factor=3,
        directory='my_dir',
        project_name=f'tuning_{representation_name.lower()}{project_suffix}',
        overwrite=True
    )

    print("\n Espace de recherche:")
    tuner.search_space_summary()

    early_stopping = tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True
    )

    print(f"\n Recherche des meilleurs hyperparamètres...")
    tuner.search(
        X_train, y_train,
        epochs=20,
        validation_data=(X_test, y_test),
        batch_size=32,
        callbacks=[early_stopping],
        verbose=1
    )

    print(f"\n Meilleurs résultats pour {representation_name}:")
    tuner.results_summary()

    best_hps = tuner.get_best_hyperparameters()[0]
    best_trial = tuner.oracle.get_best_trials(1)[0]
    best_trial_score = best_trial.score

    def get_hp_safe(name, default='N/A'):
        try:
            return best_hps.get(name)
        except:
            return default

    print(f"\n Meilleurs hyperparamètres pour {representation_name}:")
    num_layers = get_hp_safe('num_layers', 0)
    print(f"  - Nombre de couches: {num_layers}")

    if isinstance(num_layers, int):
        for i in range(num_layers):
            units = get_hp_safe(f'units_{i}')
            activation = get_hp_safe(f'activation_{i}')
            dropout = get_hp_safe(f'dropout_{i}')
            l2 = get_hp_safe(f'l2_{i}')
            print(f"  - Couche {i}: {units} unités, activation={activation}, dropout={dropout}, l2={l2}")

    lr = get_hp_safe('learning_rate', 0.001)
    if isinstance(lr, float):
        print(f"  - Learning rate: {lr:.6f}")
    else:
        print(f"  - Learning rate: {lr}")

    optimizer = get_hp_safe('optimizer', 'adam')
    print(f"  - Optimiseur: {optimizer}")

    best_model = tuner.hypermodel.build(best_hps)

    checkpoint = tf.keras.callbacks.ModelCheckpoint(
        filepath=os.path.join(tempfile.mkdtemp(), 'best_model.h5'),
        monitor='val_accuracy',
        save_best_only=True,
        mode='max'
    )
    early_stopping2 = tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True
    )

    history = best_model.fit(
        X_train, y_train,
        epochs=50,
        validation_data=(X_test, y_test),
        batch_size=32,
        callbacks=[checkpoint, early_stopping2],
        verbose=0
    )

    best_model.load_weights(checkpoint.filepath)

    y_pred_proba = best_model.predict(X_test)
    y_pred = np.argmax(y_pred_proba, axis=1)
    final_acc = accuracy_score(y_test, y_pred)

    print(f"\n Meilleure accuracy pendant le tuning: {best_trial_score:.4f}")
    print(f" Performance finale après réentraînement: {final_acc:.4f}")

    return {
        'representation': representation_name,
        'tuner': tuner,
        'best_hps': best_hps,
        'best_model': best_model,
        'history': history,
        'tuning_accuracy': best_trial_score,
        'final_accuracy': final_acc
    }

In [51]:
import scipy.sparse
import numpy as np
import joblib
from sklearn.decomposition import TruncatedSVD

# Charger les matrices sparse
X_train_ngram = scipy.sparse.load_npz('X_train_ngram.npz')
X_test_ngram = scipy.sparse.load_npz('X_test_ngram.npz')

X_train_tfidf = scipy.sparse.load_npz('X_train_tfidf.npz')
X_test_tfidf = scipy.sparse.load_npz('X_test_tfidf.npz')

# Charger les embeddings LLM
X_train_llm = np.load('X_train_llm.npy')
X_test_llm = np.load('X_test_llm.npy')

# Charger les labels
y_train = joblib.load('y_train.pkl')
y_test = joblib.load('y_test.pkl')

In [ ]:
from sklearn.decomposition import TruncatedSVD

# Réduction pour N‑Gram
svd_ngram = TruncatedSVD(n_components=256, random_state=42)

X_train_ngram_svd = svd_ngram.fit_transform(X_train_ngram)
X_test_ngram_svd = svd_ngram.transform(X_test_ngram)

# Réduction pour TF‑IDF
svd_tfidf = TruncatedSVD(n_components=256, random_state=42)

X_train_tfidf_svd = svd_tfidf.fit_transform(X_train_tfidf)
X_test_tfidf_svd = svd_tfidf.transform(X_test_tfidf)

In [52]:
# Sauvegarder les résultats
results_tuning = {}

In [56]:
results_tuning['ngram'] = tune_representation(
    X_train_ngram_svd, X_test_ngram_svd, y_train, y_test,
    representation_name='N-Gram',
    input_dim=256,
    representation_type='sparse'
)

Trial 30 Complete [00h 01m 43s]
val_accuracy: 0.829800009727478

Best val_accuracy So Far: 0.8341000080108643
Total elapsed time: 00h 25m 15s

 Meilleurs résultats pour N-Gram:
Results summary
Results in my_dir\tuning_n-gram
Showing 10 best trials
Objective(name="val_accuracy", direction="max")

Trial 0024 summary
Hyperparameters:
num_layers: 3
units_0: 640
activation_0: tanh
l2_0: 7.77064763861523e-06
dropout_0: 0.4
learning_rate: 0.00016701560845377875
units_1: 640
activation_1: tanh
l2_1: 3.4417234872201094e-05
dropout_1: 0.6000000000000001
units_2: 256
activation_2: elu
l2_2: 7.553820496028257e-05
dropout_2: 0.4
tuner/epochs: 20
tuner/initial_epoch: 7
tuner/bracket: 1
tuner/round: 1
tuner/trial_id: 0020
Score: 0.8341000080108643

Trial 0026 summary
Hyperparameters:
num_layers: 2
units_0: 512
activation_0: tanh
l2_0: 2.0337065412402656e-06
dropout_0: 0.6000000000000001
learning_rate: 6.338283748253861e-05
units_1: 1024
activation_1: elu
l2_1: 1.1534762418345611e-05
dropout_1: 0.3000

313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step

 Meilleure accuracy pendant le tuning: 0.8341
 Performance finale après réentraînement: 0.8348


In [ ]:
results_tuning['tfidf'] = tune_representation(
    X_train_tfidf_svd, X_test_tfidf_svd, y_train, y_test,
    representation_name='TF-IDF',
    input_dim=256,
    representation_type='sparse'
)

Trial 9 Complete [00h 00m 21s]
val_accuracy: 0.8461999893188477

Best val_accuracy So Far: 0.8476999998092651
Total elapsed time: 00h 08m 46s

Search: Running Trial #10

Value             |Best Value So Far |Hyperparameter
3                 |1                 |num_layers
896               |896               |units_0
relu              |elu               |activation_0
0.00026218        |7.7823e-06        |l2_0
0.5               |0.2               |dropout_0
7.5835e-05        |7.6455e-05        |learning_rate
1024              |384               |units_1
elu               |tanh              |activation_1
8.2931e-05        |2.0178e-06        |l2_1
0.3               |0.5               |dropout_1
640               |384               |units_2
elu               |elu               |activation_2
0.00013016        |0.00096496        |l2_2
0.5               |0.3               |dropout_2
3                 |3                 |tuner/epochs
0                 |0                 |tuner/initial_epoch
2  

In [38]:
results_tuning['llm'] = tune_representation(
    X_train_llm, X_test_llm, y_train, y_test,
    representation_name='LLM',
    input_dim=X_train_llm.shape[1],
    representation_type='llm'
)

Trial 30 Complete [00h 03m 14s]
val_accuracy: 0.8385999798774719

Best val_accuracy So Far: 0.8496000170707703
Total elapsed time: 00h 22m 32s

 Meilleurs résultats pour LLM:
Results summary
Results in my_dir\tuning_llm
Showing 10 best trials
Objective(name="val_accuracy", direction="max")

Trial 0017 summary
Hyperparameters:
num_layers: 3
units_0: 320
activation_0: elu
l2_0: 0.00010483434499827944
dropout_0: 0.2
learning_rate: 0.0006696330588811683
optimizer: adamax
units_1: 64
activation_1: tanh
l2_1: 3.830805835025673e-06
dropout_1: 0.5
units_2: 320
activation_2: tanh
l2_2: 0.0002510676487272628
dropout_2: 0.5
units_3: 512
activation_3: elu
l2_3: 0.0003715679781753188
dropout_3: 0.4
tuner/epochs: 20
tuner/initial_epoch: 7
tuner/bracket: 2
tuner/round: 2
tuner/trial_id: 0013
Score: 0.8496000170707703

Trial 0024 summary
Hyperparameters:
num_layers: 2
units_0: 192
activation_0: tanh
l2_0: 3.1785936866372997e-06
dropout_0: 0.5
learning_rate: 0.0010854297189198875
optimizer: adam
units_

313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step

✅ Meilleure accuracy pendant le tuning: 0.8496
✅ Performance finale après réentraînement: 0.8486


In [ ]:
import pickle
from tensorflow import keras
import os

save_dir = "../Sauvegarde"
os.makedirs(save_dir, exist_ok=True)

results_light = {}
for key, res in results_tuning.items():
    model_filename = f'best_model_{key}.keras'
    model_path = os.path.join(save_dir, model_filename)
    res['best_model'].save(model_path)  
    results_light[key] = {
        'representation': res['representation'],
        'best_hps': res['best_hps'],
        'history': res['history'].history,
        'tuning_accuracy': res['tuning_accuracy'],
        'final_accuracy': res['final_accuracy'],
        'model_path': model_filename  
    }

# Sauvegarde du dictionnaire léger
with open(os.path.join(save_dir, 'results_tuning.pkl'), 'wb') as f:
    pickle.dump(results_light, f)

print(f"Résultats deep learning sauvegardés dans {save_dir}")

In [42]:
# Afficher la comparaison
def safe_get_hp(hps, key, default='N/A'):
    try:
        return hps.get(key)
    except:
        return default

comparison_rows = []
for rep_name, res in results_tuning.items():
    row = {
        'Représentation': res['representation'],
        'Tuning accuracy': f"{res['tuning_accuracy']:.4f}",
        'Final accuracy': f"{res['final_accuracy']:.4f}",
        'Nombre de couches': safe_get_hp(res['best_hps'], 'num_layers'),
        'Learning rate': f"{safe_get_hp(res['best_hps'], 'learning_rate', 0):.6f}",
        'Optimiseur': safe_get_hp(res['best_hps'], 'optimizer', 'adam')
    }
    comparison_rows.append(row)

comparison_df = pd.DataFrame(comparison_rows)
print("\n" + "="*70)
print("COMPARAISON DES MEILLEURES CONFIGURATIONS")
print("="*70)
print(comparison_df.to_string(index=False))


COMPARAISON DES MEILLEURES CONFIGURATIONS
Représentation Tuning accuracy Final accuracy  Nombre de couches Learning rate Optimiseur
        N-Gram          0.8381         0.8343                  3      0.000288       adam
        TF-IDF          0.8529         0.8496                  3      0.000079       adam
           LLM          0.8496         0.8486                  3      0.000670     adamax


---
## Transformer

In [ ]:
!pip install transformers datasets accelerate
!pip install accelerate>=1.1.0

In [24]:
from sklearn.metrics import accuracy_score, classification_report
import torch
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification, Trainer, TrainingArguments
from transformers import EarlyStoppingCallback
from datasets import Dataset
import pandas as pd
from sklearn.model_selection import train_test_split

In [28]:
# Sample for training
SAMPLE_SIZE = 10000
df_sample = df.sample(min(SAMPLE_SIZE, len(df)), random_state=42)
print(f"Using {len(df_sample)} samples")

# Polarity mapping
polarity_map = {'negative': 0, 'neutral': 1, 'positive': 2}
df_sample['polarity_num'] = df_sample['polarity'].map(polarity_map)

# Split
X = df_sample['text']
y = df_sample['polarity_num']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {len(X_train)}, Test: {len(X_test)}")

Using 10000 samples
Train: 8000, Test: 2000


In [30]:

# Tokenizer
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

def tokenize_function(examples):
    return tokenizer(examples['text'], padding='max_length', truncation=True, max_length=128)

# Créer des datasets Hugging Face
train_dataset = Dataset.from_dict({'text': X_train, 'label': y_train})
test_dataset = Dataset.from_dict({'text': X_test, 'label': y_test})

train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

# Format pour PyTorch
train_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
test_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])

Map: 100%|██████████| 2000/2000 [00:00<00:00, 7073.55 examples/s]


In [31]:
model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=3)

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=100,
    eval_strategy="epoch",          
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    return {'accuracy': acc}

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 501.04it/s, Materializing param=distilbert.transformer.layer.5.sa_layer_norm.weight]   
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_D

In [32]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

trainer.train()

c:\Users\anas\Desktop\Cours\SAE\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy
1,0.410861,0.388855,0.858000
2,0.353313,0.371922,0.847500
3,0.189838,0.459442,0.864000


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.36it/s]
c:\Users\anas\Desktop\Cours\SAE\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.61it/s]
c:\Users\anas\Desktop\Cours\SAE\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.97it/s]
There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=1500, training_loss=0.35904381561279297, metrics={'train_runtime': 5589.298, 'train_samples_per_second': 4.294, 'train_steps_per_second': 0.268, 'total_flos': 794818566144000.0, 'train_loss': 0.35904381561279297, 'epoch': 3.0})

In [33]:
predictions = trainer.predict(test_dataset)
y_pred_transformer = np.argmax(predictions.predictions, axis=1)
acc_transformer = accuracy_score(y_test, y_pred_transformer)
print(f"Transformer Accuracy: {acc_transformer:.4f}")

c:\Users\anas\Desktop\Cours\SAE\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Transformer Accuracy: 0.8640


In [37]:
import os
import json

save_dir = "../Sauvegarde"
os.makedirs(save_dir, exist_ok=True)

# Sauvegarde du modèle et du tokenizer
model.save_pretrained(os.path.join(save_dir, 'distilbert_model'))
tokenizer.save_pretrained(os.path.join(save_dir, 'distilbert_tokenizer'))

# Sauvegarde de l'accuracy et éventuellement des prédictions
with open(os.path.join(save_dir, 'transformer_metadata.json'), 'w') as f:
    json.dump({'accuracy': acc_transformer}, f)

print("Modèle Transformer sauvegardé.")

Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.61s/it]

Modèle Transformer sauvegardé.


---
## 6. Results Summary

In [ ]:
import os
import joblib
import numpy as np
import pickle
from tensorflow import keras
import pandas as pd
from sklearn.metrics import accuracy_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

save_dir = "../Sauvegarde"

# --- Recharger les résultats des modèles classiques ---
results = joblib.load(os.path.join(save_dir, 'results_ml.pkl'))

# --- Recharger les modèles classiques ---
# N-Gram
lr_ngram = joblib.load(os.path.join(save_dir, 'lr_ngram.pkl'))
svm_ngram = joblib.load(os.path.join(save_dir, 'svm_ngram.pkl'))
nb_ngram = joblib.load(os.path.join(save_dir, 'nb_ngram.pkl'))
rf_ngram = joblib.load(os.path.join(save_dir, 'rf_ngram.pkl'))

# TF-IDF
lr_tfidf = joblib.load(os.path.join(save_dir, 'lr_tfidf.pkl'))
svm_tfidf = joblib.load(os.path.join(save_dir, 'svm_tfidf.pkl'))
nb_tfidf = joblib.load(os.path.join(save_dir, 'nb_tfidf.pkl'))
rf_tfidf = joblib.load(os.path.join(save_dir, 'rf_tfidf.pkl'))

# LLM
lr_llm = joblib.load(os.path.join(save_dir, 'lr_llm.pkl'))
svm_llm = joblib.load(os.path.join(save_dir, 'svm_llm.pkl'))
rf_llm = joblib.load(os.path.join(save_dir, 'rf_llm.pkl'))

# --- Recharger les données de test ---
X_test_ngram = np.load(os.path.join(save_dir, 'X_test_ngram.npy'))
X_test_tfidf = np.load(os.path.join(save_dir, 'X_test_tfidf.npy'))
X_test_llm = np.load(os.path.join(save_dir, 'X_test_llm.npy'))
y_test = joblib.load(os.path.join(save_dir, 'y_test.pkl'))

# --- Recharger les résultats du deep learning ---
with open(os.path.join(save_dir, 'results_tuning.pkl'), 'rb') as f:
    results_loaded = pickle.load(f)

# Recharger les modèles deep learning
best_model_ngram = keras.models.load_model(os.path.join(save_dir, results_loaded['ngram']['model_path']))
best_model_tfidf = keras.models.load_model(os.path.join(save_dir, results_loaded['tfidf']['model_path']))
best_model_llm = keras.models.load_model(os.path.join(save_dir, results_loaded['llm']['model_path']))

# Récupérer les accuracies finales
acc_ngram_dl = results_loaded['ngram']['final_accuracy']
acc_tfidf_dl = results_loaded['tfidf']['final_accuracy']
acc_llm_dl = results_loaded['llm']['final_accuracy']

print("Rechargement terminé depuis le dossier ../Sauvegarde/.")

In [ ]:
# Construire le DataFrame complet 
all_results = results.copy()

# Ajouter les MLP
all_results.append({'Model': 'MLP (N-Gram)', 'Representation': 'N-Gram', 'Accuracy': acc_ngram_dl})
all_results.append({'Model': 'MLP (TF-IDF)', 'Representation': 'TF-IDF', 'Accuracy': acc_tfidf_dl})
all_results.append({'Model': 'MLP (LLM)', 'Representation': 'LLM', 'Accuracy': acc_llm_dl})

# Ajouter le Transformer si vous l'avez sauvegardé et rechargé
# Par exemple, si vous avez un modèle Transformer et son accuracy :
# all_results.append({'Model': 'DistilBERT', 'Representation': 'Transformer', 'Accuracy': acc_transformer})

results_df = pd.DataFrame(all_results)
print(results_df.sort_values('Accuracy', ascending=False).to_string(index=False))


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# --- Graphique en barres ---
plt.figure(figsize=(14, 6))
sns.barplot(data=results_df, x='Model', y='Accuracy', hue='Representation', palette='Set2')
plt.title('Comparaison de tous les modèles (ML, MLP, Transformer)')
plt.xticks(rotation=45, ha='right')
plt.ylim(0.5, 1.0)
plt.legend(title='Représentation', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig(os.path.join(save_dir, 'fig_all_models_comparison.png'), dpi=150)
plt.show()

## 7. Confusion Matrices

In [ ]:
from sklearn.metrics import confusion_matrix

# --- Matrices de confusion pour les meilleurs modèles ---
# Meilleur ML (ex: SVM TF-IDF)
y_pred_ml = svm_tfidf.predict(X_test_tfidf)
acc_ml = accuracy_score(y_test, y_pred_ml)

# Meilleur MLP (ex: TF-IDF)
y_pred_mlp = np.argmax(best_model_tfidf.predict(X_test_tfidf), axis=1)
acc_mlp = accuracy_score(y_test, y_pred_mlp)

# Si vous avez un Transformer :
# y_pred_transformer = ...  # à calculer
# acc_trans = accuracy_score(y_test, y_pred_transformer)

fig, axes = plt.subplots(1, 3 if 'y_pred_transformer' in locals() else 2, figsize=(15, 5))

# ML
cm_ml = confusion_matrix(y_test, y_pred_ml)
sns.heatmap(cm_ml, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Neg', 'Neu', 'Pos'], yticklabels=['Neg', 'Neu', 'Pos'])
axes[0].set_title(f'ML (SVM TF-IDF) - Acc: {acc_ml:.4f}')
axes[0].set_xlabel('Prédit'); axes[0].set_ylabel('Réel')

# MLP
cm_mlp = confusion_matrix(y_test, y_pred_mlp)
sns.heatmap(cm_mlp, annot=True, fmt='d', cmap='Greens', ax=axes[1],
            xticklabels=['Neg', 'Neu', 'Pos'], yticklabels=['Neg', 'Neu', 'Pos'])
axes[1].set_title(f'MLP (TF-IDF) - Acc: {acc_mlp:.4f}')
axes[1].set_xlabel('Prédit'); axes[1].set_ylabel('Réel')

# Transformer (si disponible)
if 'y_pred_transformer' in locals():
    cm_trans = confusion_matrix(y_test, y_pred_transformer)
    sns.heatmap(cm_trans, annot=True, fmt='d', cmap='Oranges', ax=axes[2],
                xticklabels=['Neg', 'Neu', 'Pos'], yticklabels=['Neg', 'Neu', 'Pos'])
    axes[2].set_title(f'DistilBERT - Acc: {acc_trans:.4f}')
    axes[2].set_xlabel('Prédit'); axes[2].set_ylabel('Réel')

plt.tight_layout()
plt.savefig(os.path.join(save_dir, 'fig_confusion_matrices_all.png'), dpi=150)
plt.show()

---
# Prédire la Note


In [ ]:
SAMPLE_SIZE = 100               
TEST_SIZE = 0.2
RANDOM_STATE = 42
save_dir = "../Sauvegarde_Note"
os.makedirs(save_dir, exist_ok=True)

# 1. Charger les données
df = pd.read_parquet("../Data/prepared_reviews.parquet")
df_sample = df.sample(SAMPLE_SIZE, random_state=RANDOM_STATE)

# 2. Préparer les labels de note 
y_rating = df_sample['stars'] - 1   

# 3. Split train/test
X_train_Note, X_test_Note, y_train_Note, y_test_Note = train_test_split(
    df_sample['text'], y_rating,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE
)
print(f"Train: {len(X_train_Note)}, Test: {len(X_test_Note)}")

---
# Fichier Test : prédire la Note + polarité
